In [2]:
# ============================================================
# FASE 3 — Setup: imports, paths e constantes
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm.auto import tqdm

# ============================================================
# Paths e constantes
# ============================================================

BASE_ANONIMIZADO = "/content/drive/MyDrive/Mestrado/Dados_Anonimizados"
BASE_OUT_FASE3   = "/content/drive/MyDrive/Mestrado/Resultados_Fase3"

os.makedirs(BASE_OUT_FASE3, exist_ok=True)

LAT_COL = "latitude"
LON_COL = "longitude"

ANOS = [2019, 2020, 2021, 2022, 2023]  # mesma cobertura da Fase 2

# Técnicas: original é referência (não atacamos),
# generalizacao_dec1 foi excluída na decisão da Fase 2.
TECNICAS_ATACAR = [
    "permutacao",
    "generalizacao_dec2",
    "microagregacao_k2", "microagregacao_k5", "microagregacao_k10",
    "dp_eps_0.1", "dp_eps_0.5", "dp_eps_1.0", "dp_eps_2.0", "dp_eps_5.0",
]

TECNICAS_ESTOCASTICAS = {
    "permutacao",
    "microagregacao_k2", "microagregacao_k5", "microagregacao_k10",
    "dp_eps_0.1", "dp_eps_0.5", "dp_eps_1.0", "dp_eps_2.0", "dp_eps_5.0",
}

# Configuração da Fase 3
N_SEEDS_USAR     = 10                    # mesmas seeds da Fase 2
NIVEIS_AUX       = [0.01, 0.05, 0.10]    # 1%, 5%, 10%
RAIOS_A1_METROS  = [50, 100, 200]
KS_A2            = [3, 5, 10]
RAIOS_A3_METROS  = [50, 100, 200, 500]
JANELA_A4_DIAS   = 7
SEED_AMOSTRAGEM_AUX = 12345              # fixa para reprodutibilidade

print(f"✅ Setup Fase 3 concluído.")
print(f"   Técnicas a atacar: {len(TECNICAS_ATACAR)} ({len(TECNICAS_ESTOCASTICAS)} estocásticas)")
print(f"   Seeds: {N_SEEDS_USAR}")
print(f"   Anos: {ANOS}")
print(f"   Ataques: A1 (NN-Linkage), A2 (Top-k), A3 (Ambiguidade), A4 (Reconstrução temporal)")
print(f"   Níveis de aux: {[f'{n*100:.0f}%' for n in NIVEIS_AUX]}")
print(f"   Saída: {BASE_OUT_FASE3}")

# Estimativa preliminar do total de avaliações
n_estoc = len(TECNICAS_ESTOCASTICAS) * N_SEEDS_USAR
n_det = len(TECNICAS_ATACAR) - len(TECNICAS_ESTOCASTICAS)
n_combinacoes = (n_estoc + n_det) * len(NIVEIS_AUX)
n_a1 = n_combinacoes * len(RAIOS_A1_METROS)
n_a2 = n_combinacoes * len(KS_A2)
n_a3_pontos = n_combinacoes * len(RAIOS_A3_METROS)  # A3 não tem aux
n_a4 = n_estoc + n_det  # A4 não depende de aux nem de raio único

print(f"\n📊 Volume estimado:")
print(f"   Combinações técnica×seed×nivel_aux: {n_combinacoes}")
print(f"   A1 (×raio): {n_a1}")
print(f"   A2 (×k): {n_a2}")
print(f"   A3 (×raio): {n_a3_pontos}")
print(f"   A4 (×janela): {n_a4}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Setup Fase 3 concluído.
   Técnicas a atacar: 10 (9 estocásticas)
   Seeds: 10
   Anos: [2019, 2020, 2021, 2022, 2023]
   Ataques: A1 (NN-Linkage), A2 (Top-k), A3 (Ambiguidade), A4 (Reconstrução temporal)
   Níveis de aux: ['1%', '5%', '10%']
   Saída: /content/drive/MyDrive/Mestrado/Resultados_Fase3

📊 Volume estimado:
   Combinações técnica×seed×nivel_aux: 273
   A1 (×raio): 819
   A2 (×k): 819
   A3 (×raio): 1092
   A4 (×janela): 91


In [3]:
# ============================================================
# Funções auxiliares: haversine, amostragem D_aux, carregamento
# ============================================================

def haversine_m(lat1, lon1, lat2, lon2):
    """
    Distância haversine em metros entre dois pontos (ou arrays de pontos).
    Aceita escalares ou arrays NumPy. Broadcasting funciona.
    """
    R = 6_371_000.0
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def carregar_par_orig_anon(tecnica, seed, anos=ANOS, base=BASE_ANONIMIZADO):
    """
    Carrega o par (original, anonimizado) para uma técnica×seed.
    Garante alinhamento por índice de linha — registros na mesma
    posição em df_orig e df_anon correspondem ao MESMO registro
    pré e pós anonimização (preservado pela Fase 1).

    Retorna (df_orig, df_anon) ambos com colunas latitude, longitude,
    inspection_realized_at.
    """
    dfs_orig, dfs_anon = [], []
    for ano in anos:
        # Original: seed=0 sempre
        path_o = f"{base}/original/{ano}.parquet"
        df_o = pd.read_parquet(path_o)
        df_o = df_o[df_o["seed"] == 0].reset_index(drop=True)

        # Anonimizado: seed especificada
        path_a = f"{base}/{tecnica}/{ano}.parquet"
        df_a = pd.read_parquet(path_a)
        df_a = df_a[df_a["seed"] == seed].reset_index(drop=True)

        # Sanidade: mesmo tamanho garante alinhamento
        assert len(df_o) == len(df_a), (
            f"Tamanhos diferentes em {ano}: orig={len(df_o)}, "
            f"anon={tecnica}/seed={seed}={len(df_a)}"
        )

        dfs_orig.append(df_o)
        dfs_anon.append(df_a)

    df_orig = pd.concat(dfs_orig, ignore_index=True)
    df_anon = pd.concat(dfs_anon, ignore_index=True)

    cols_uteis = [LAT_COL, LON_COL, "inspection_realized_at"]
    return df_orig[cols_uteis], df_anon[cols_uteis]


def amostrar_d_aux(n_total, fracao, seed_amostragem=SEED_AMOSTRAGEM_AUX):
    """
    Sorteia índices para D_aux (subconjunto auxiliar do adversário).

    Parâmetros:
        n_total : tamanho de D_original
        fracao  : 0.01, 0.05, 0.10 etc.
        seed_amostragem : SEED FIXA para reprodutibilidade — não
                          confundir com seed da técnica de anonimização.

    Retorna array de índices.
    """
    rng = np.random.default_rng(seed_amostragem)
    n_aux = int(round(n_total * fracao))
    return rng.choice(n_total, size=n_aux, replace=False)


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste: carregar par (original, permutacao seed=0)\n")

df_orig, df_anon = carregar_par_orig_anon("permutacao", seed=0)
print(f"   Tamanho: orig={len(df_orig):,}, anon={len(df_anon):,}")

# Verifica que estão diferentes (permutação mudou as coords)
diff_lat = (df_orig[LAT_COL].values != df_anon[LAT_COL].values).sum()
print(f"   Linhas com lat diferente após permutacao: {diff_lat:,}")

# Haversine entre pontos correspondentes
dist = haversine_m(
    df_orig[LAT_COL].values, df_orig[LON_COL].values,
    df_anon[LAT_COL].values, df_anon[LON_COL].values
)
print(f"   Deslocamento permutacao seed=0:")
print(f"      mediana = {np.median(dist):,.0f}m")
print(f"      máximo  = {dist.max():,.0f}m")

# Teste D_aux
idx_aux = amostrar_d_aux(len(df_orig), 0.01)
print(f"\n   D_aux 1%: {len(idx_aux):,} índices sorteados")
print(f"   Primeiros 5 índices: {idx_aux[:5]}")

print(f"\n✅ Funções auxiliares OK.")

🧪 Teste: carregar par (original, permutacao seed=0)

   Tamanho: orig=235,707, anon=235,707
   Linhas com lat diferente após permutacao: 233,666
   Deslocamento permutacao seed=0:
      mediana = 6,754m
      máximo  = 23,691m

   D_aux 1%: 2,357 índices sorteados
   Primeiros 5 índices: [27881 16402 85940 10058 63666]

✅ Funções auxiliares OK.


In [4]:
# ============================================================
# Ataque A1 — Reidentificação direta por vizinhança espacial
# Sucesso: distância(r_aux, M(r_aux)) <= R   (match VERDADEIRO)
# ============================================================

def ataque_a1(df_orig, df_anon, idx_aux, raios_m):
    """
    Para cada r_aux em D_aux, mede distância haversine até seu correspondente
    anonimizado M(r_aux). Sucesso se distância <= R.

    Crítico: M(r_aux) é o registro MESMO ÍNDICE em df_anon (pareamento
    preservado pela Fase 1).

    Retorna dict: {raio: taxa_sucesso}.
    """
    lat_aux = df_orig[LAT_COL].values[idx_aux]
    lon_aux = df_orig[LON_COL].values[idx_aux]
    lat_anon = df_anon[LAT_COL].values[idx_aux]
    lon_anon = df_anon[LON_COL].values[idx_aux]

    # Distância entre r_aux e seu correspondente anonimizado
    dist = haversine_m(lat_aux, lon_aux, lat_anon, lon_anon)

    taxas = {}
    for R in raios_m:
        taxas[R] = float((dist <= R).mean())
    return taxas


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste A1: permutacao seed=0, aux=10%\n")

df_orig, df_anon = carregar_par_orig_anon("permutacao", seed=0)
idx_aux = amostrar_d_aux(len(df_orig), 0.10)
taxas = ataque_a1(df_orig, df_anon, idx_aux, RAIOS_A1_METROS)

print(f"   Taxa de sucesso A1 (permutacao, aux=10%):")
for R, t in taxas.items():
    print(f"      R={R}m: {t*100:5.2f}%")

# Sanity: original deve ter 100% (distância 0)
df_orig2, df_anon_or = carregar_par_orig_anon("dp_eps_5.0", seed=0)
idx_aux = amostrar_d_aux(len(df_orig2), 0.10)
taxas_dp5 = ataque_a1(df_orig2, df_anon_or, idx_aux, RAIOS_A1_METROS)
print(f"\n   Taxa de sucesso A1 (dp_eps_5.0, aux=10%):")
for R, t in taxas_dp5.items():
    print(f"      R={R}m: {t*100:5.2f}%")

print(f"\n✅ A1 OK.")

🧪 Teste A1: permutacao seed=0, aux=10%

   Taxa de sucesso A1 (permutacao, aux=10%):
      R=50m:  0.00%
      R=100m:  0.02%
      R=200m:  0.05%

   Taxa de sucesso A1 (dp_eps_5.0, aux=10%):
      R=50m:  2.79%
      R=100m:  9.45%
      R=200m: 26.63%

✅ A1 OK.


In [5]:
# ============================================================
# Ataque A2 — Ligação probabilística Top-k
# Sucesso: M(r_aux) está entre os k vizinhos mais próximos de r_aux
#          em D_anon, segundo distância haversine
# ============================================================

from sklearn.neighbors import NearestNeighbors


def ataque_a2(df_orig, df_anon, idx_aux, ks):
    """
    Para cada r_aux, calcula os k vizinhos mais próximos em D_anon e
    verifica se M(r_aux) (o correspondente anonimizado, no MESMO índice)
    está entre eles.

    Retorna dict: {k: taxa_sucesso}.
    """
    # Coordenadas (usando radianos para o BallTree haversine)
    coords_anon_rad = np.deg2rad(df_anon[[LAT_COL, LON_COL]].values)
    coords_aux_rad  = np.deg2rad(df_orig.iloc[idx_aux][[LAT_COL, LON_COL]].values)

    k_max = max(ks)

    # NearestNeighbors com métrica haversine (precisa radianos)
    nn = NearestNeighbors(n_neighbors=k_max, metric='haversine', n_jobs=-1)
    nn.fit(coords_anon_rad)

    # Para cada r_aux: índices dos k_max vizinhos mais próximos em df_anon
    _, idx_vizinhos = nn.kneighbors(coords_aux_rad)
    # idx_vizinhos shape: (len(idx_aux), k_max)

    # Sucesso: o correspondente verdadeiro M(r_aux) é idx_aux[i]
    # (mesma posição em df_anon)
    taxas = {}
    for k in ks:
        # Para os k primeiros vizinhos, verifica se idx_aux[i] está lá
        viz_k = idx_vizinhos[:, :k]
        sucesso = np.array([
            idx_aux[i] in viz_k[i]
            for i in range(len(idx_aux))
        ])
        taxas[k] = float(sucesso.mean())
    return taxas


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste A2: permutacao seed=0, aux=10%\n")

df_orig, df_anon = carregar_par_orig_anon("permutacao", seed=0)
idx_aux = amostrar_d_aux(len(df_orig), 0.10)
taxas = ataque_a2(df_orig, df_anon, idx_aux, KS_A2)

print(f"   Probabilidade Top-k (permutacao, aux=10%):")
for k, t in taxas.items():
    print(f"      k={k:2d}: {t*100:5.2f}%")

df_orig2, df_anon_or = carregar_par_orig_anon("dp_eps_5.0", seed=0)
idx_aux = amostrar_d_aux(len(df_orig2), 0.10)
taxas_dp5 = ataque_a2(df_orig2, df_anon_or, idx_aux, KS_A2)
print(f"\n   Probabilidade Top-k (dp_eps_5.0, aux=10%):")
for k, t in taxas_dp5.items():
    print(f"      k={k:2d}: {t*100:5.2f}%")

print(f"\n✅ A2 OK.")

🧪 Teste A2: permutacao seed=0, aux=10%

   Probabilidade Top-k (permutacao, aux=10%):
      k= 3:  0.01%
      k= 5:  0.01%
      k=10:  0.01%

   Probabilidade Top-k (dp_eps_5.0, aux=10%):
      k= 3:  0.62%
      k= 5:  0.98%
      k=10:  1.79%

✅ A2 OK.


In [6]:
# ============================================================
# Ataque A3 — Anonimato efetivo por ambiguidade espacial
# Para cada r* em D_anon, conta quantos pontos de D_orig estão
# dentro de raio R. Valores maiores = mais ambiguidade = mais privacidade.
# ============================================================

from sklearn.neighbors import BallTree


def ataque_a3(df_orig, df_anon, raios_m, amostra_n=None, seed_amostra=42):
    """
    Para cada r* em D_anon (ou subamostra), conta vizinhos de D_orig
    dentro de cada raio. Retorna ambiguidade média.

    Parâmetros:
        amostra_n : se não None, amostra esse número de pontos de D_anon
                    para acelerar (cálculo é O(n^2) para todo D_anon).
                    Padrão None usa todos os pontos.

    Retorna dict: {raio: ambiguidade_media}.
    """
    coords_orig_rad = np.deg2rad(df_orig[[LAT_COL, LON_COL]].values)
    coords_anon_rad = np.deg2rad(df_anon[[LAT_COL, LON_COL]].values)

    # Amostragem opcional de D_anon
    if amostra_n is not None and amostra_n < len(coords_anon_rad):
        rng = np.random.default_rng(seed_amostra)
        idx_amostra = rng.choice(len(coords_anon_rad), size=amostra_n, replace=False)
        coords_anon_rad = coords_anon_rad[idx_amostra]

    # BallTree com métrica haversine; raio em radianos
    R_earth_m = 6_371_000.0
    tree = BallTree(coords_orig_rad, metric='haversine')

    ambiguidades = {}
    for R_m in raios_m:
        R_rad = R_m / R_earth_m
        # query_radius retorna lista de arrays (vizinhos por ponto)
        counts = tree.query_radius(coords_anon_rad, r=R_rad, count_only=True)
        ambiguidades[R_m] = float(counts.mean())
    return ambiguidades


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste A3: ambiguidade média por raio (subamostra 5k pontos)\n")

for tecnica_teste in ["permutacao", "dp_eps_0.1", "dp_eps_5.0", "generalizacao_dec2"]:
    df_orig, df_anon = carregar_par_orig_anon(tecnica_teste, seed=0)
    ambig = ataque_a3(df_orig, df_anon, RAIOS_A3_METROS, amostra_n=5000)

    print(f"\n   {tecnica_teste}:")
    for R, a in ambig.items():
        print(f"      R={R:3d}m: {a:7.1f} candidatos médios")

print(f"\n✅ A3 OK.")

🧪 Teste A3: ambiguidade média por raio (subamostra 5k pontos)


   permutacao:
      R= 50m:    16.1 candidatos médios
      R=100m:    43.7 candidatos médios
      R=200m:   149.8 candidatos médios
      R=500m:   999.1 candidatos médios

   dp_eps_0.1:
      R= 50m:     1.4 candidatos médios
      R=100m:     6.0 candidatos médios
      R=200m:    25.5 candidatos médios
      R=500m:   151.8 candidatos médios

   dp_eps_5.0:
      R= 50m:    18.6 candidatos médios
      R=100m:    75.0 candidatos médios
      R=200m:   282.0 candidatos médios
      R=500m:  1682.0 candidatos médios

   generalizacao_dec2:
      R= 50m:    26.6 candidatos médios
      R=100m:    72.9 candidatos médios
      R=200m:   257.9 candidatos médios
      R=500m:  1712.3 candidatos médios

✅ A3 OK.


In [7]:
# ============================================================
# Ataque A4 — Reconstrução temporal por agregação
# Para registros do MESMO PONTO REAL com múltiplas inspeções, agrega
# (mediana espacial) as posições anonimizadas e mede erro vs verdade.
# ============================================================

def ataque_a4(df_orig, df_anon, janela_dias=JANELA_A4_DIAS, raios_m=(50, 100, 200)):
    """
    Identifica registros do mesmo ponto físico (mesma latitude/longitude
    em df_orig) e agrega suas observações anonimizadas pela mediana
    espacial. Mede distância entre a estimativa e a verdade.

    Lógica:
        1. Agrupa em df_orig por (lat, lon) → cada grupo é um ponto físico
           com múltiplas inspeções no tempo.
        2. Para cada grupo com >= 2 inspeções, calcula:
              - posição verdadeira: (lat, lon) original
              - estimativa: mediana espacial de M(r_t) entre as inspeções
        3. Distância entre verdade e estimativa.

    Retorna dict com erro mediano global + taxas de sucesso por raio.
    """
    # Garante que df_orig e df_anon têm o mesmo índice
    df_o = df_orig.reset_index(drop=True)
    df_a = df_anon.reset_index(drop=True)

    # Agrupa pontos físicos por (lat, lon) ORIGINAIS
    grupos = df_o.groupby([LAT_COL, LON_COL]).indices  # dict: (lat,lon) -> array de índices

    erros = []
    for (lat_v, lon_v), idxs in grupos.items():
        if len(idxs) < 2:
            continue  # ponto inspecionado só uma vez, A4 não se aplica

        # Coords anonimizadas para esse ponto físico
        lats_a = df_a[LAT_COL].values[idxs]
        lons_a = df_a[LON_COL].values[idxs]

        # Estimativa: mediana espacial
        lat_hat = np.median(lats_a)
        lon_hat = np.median(lons_a)

        # Erro: distância entre estimativa e verdade
        err = haversine_m(lat_v, lon_v, lat_hat, lon_hat)
        erros.append(float(err))

    erros = np.array(erros)
    if len(erros) == 0:
        return {'n_pontos_validos': 0, 'erro_mediano_m': np.nan,
                'erro_p95_m': np.nan, **{f'sucesso_R{R}': np.nan for R in raios_m}}

    res = {
        'n_pontos_validos': len(erros),
        'erro_mediano_m':   float(np.median(erros)),
        'erro_p95_m':       float(np.percentile(erros, 95)),
    }
    for R in raios_m:
        res[f'sucesso_R{R}'] = float((erros <= R).mean())
    return res


# ─── Validação rápida ──────────────────────────────────────────
print("🧪 Teste A4: reconstrução temporal\n")

for tecnica_teste in ["permutacao", "dp_eps_1.0", "dp_eps_5.0", "generalizacao_dec2"]:
    df_orig, df_anon = carregar_par_orig_anon(tecnica_teste, seed=0)
    res = ataque_a4(df_orig, df_anon)

    print(f"\n   {tecnica_teste}:")
    print(f"      Pontos físicos com ≥2 inspeções: {res['n_pontos_validos']:,}")
    print(f"      Erro mediano: {res['erro_mediano_m']:,.0f}m")
    print(f"      Erro P95:     {res['erro_p95_m']:,.0f}m")
    for R in (50, 100, 200):
        print(f"      Sucesso R={R}m: {res[f'sucesso_R{R}']*100:5.2f}%")

print(f"\n✅ A4 OK.")

🧪 Teste A4: reconstrução temporal


   permutacao:
      Pontos físicos com ≥2 inspeções: 2,447
      Erro mediano: 4,745m
      Erro P95:     10,725m
      Sucesso R=50m:  0.00%
      Sucesso R=100m:  0.00%
      Sucesso R=200m:  0.04%

   dp_eps_1.0:
      Pontos físicos com ≥2 inspeções: 2,447
      Erro mediano: 205m
      Erro P95:     704m
      Sucesso R=50m:  5.15%
      Sucesso R=100m: 18.02%
      Sucesso R=200m: 48.51%

   dp_eps_5.0:
      Pontos físicos com ≥2 inspeções: 2,447
      Erro mediano: 41m
      Erro P95:     141m
      Sucesso R=50m: 61.01%
      Sucesso R=100m: 89.33%
      Sucesso R=200m: 97.87%

   generalizacao_dec2:
      Pontos físicos com ≥2 inspeções: 2,447
      Erro mediano: 401m
      Erro P95:     627m
      Sucesso R=50m:  1.06%
      Sucesso R=100m:  2.94%
      Sucesso R=200m: 11.07%

✅ A4 OK.


In [8]:
# ============================================================
# Loop principal: varre técnicas × seeds × níveis_aux × ataques
# Salva em /Resultados_Fase3/resultados_ataques.parquet
# Reentrante: checkpoint após cada técnica×seed
# ============================================================

CAMINHO_ATAQUES = f"{BASE_OUT_FASE3}/resultados_ataques.parquet"


def executar_todos_ataques(df_orig, df_anon, tecnica, seed, niveis_aux):
    """
    Executa A1, A2, A3, A4 para uma técnica×seed.

    Retorna lista de dicts (linhas para o DataFrame final).
    """
    linhas = []
    n_total = len(df_orig)

    # ── A3 e A4 não dependem de aux ──
    ambig = ataque_a3(df_orig, df_anon, RAIOS_A3_METROS, amostra_n=5000)
    for R, a in ambig.items():
        linhas.append({
            'tecnica': tecnica, 'seed': seed, 'ataque': 'A3',
            'parametro': 'raio_m', 'valor_param': R,
            'metrica': 'ambiguidade_media', 'valor': a,
            'nivel_aux': None,
        })

    a4_res = ataque_a4(df_orig, df_anon)
    linhas.append({
        'tecnica': tecnica, 'seed': seed, 'ataque': 'A4',
        'parametro': 'janela_dias', 'valor_param': JANELA_A4_DIAS,
        'metrica': 'erro_mediano_m', 'valor': a4_res['erro_mediano_m'],
        'nivel_aux': None,
    })
    linhas.append({
        'tecnica': tecnica, 'seed': seed, 'ataque': 'A4',
        'parametro': 'janela_dias', 'valor_param': JANELA_A4_DIAS,
        'metrica': 'erro_p95_m', 'valor': a4_res['erro_p95_m'],
        'nivel_aux': None,
    })

    # ── A1 e A2 variam por nível auxiliar ──
    for nivel in niveis_aux:
        idx_aux = amostrar_d_aux(n_total, nivel)

        # A1
        a1_taxas = ataque_a1(df_orig, df_anon, idx_aux, RAIOS_A1_METROS)
        for R, t in a1_taxas.items():
            linhas.append({
                'tecnica': tecnica, 'seed': seed, 'ataque': 'A1',
                'parametro': 'raio_m', 'valor_param': R,
                'metrica': 'taxa_sucesso', 'valor': t,
                'nivel_aux': nivel,
            })

        # A2
        a2_taxas = ataque_a2(df_orig, df_anon, idx_aux, KS_A2)
        for k, t in a2_taxas.items():
            linhas.append({
                'tecnica': tecnica, 'seed': seed, 'ataque': 'A2',
                'parametro': 'top_k', 'valor_param': k,
                'metrica': 'taxa_sucesso', 'valor': t,
                'nivel_aux': nivel,
            })

    return linhas


# ────────────────────────────────────────────────────────────────
# Execução com checkpoint
# ────────────────────────────────────────────────────────────────

# Reentrante
if os.path.exists(CAMINHO_ATAQUES):
    df_prev = pd.read_parquet(CAMINHO_ATAQUES)
    feitas = set(zip(df_prev['tecnica'], df_prev['seed']))
    print(f"📂 Checkpoint: {len(feitas)} combinações técnica×seed já feitas")
else:
    df_prev = pd.DataFrame()
    feitas = set()
    print(f"🆕 Sem checkpoint, começando do zero")

# Combinações pendentes
combinacoes = []
for tec in TECNICAS_ATACAR:
    n_seeds_tec = N_SEEDS_USAR if tec in TECNICAS_ESTOCASTICAS else 1
    for seed in range(n_seeds_tec):
        if (tec, seed) not in feitas:
            combinacoes.append((tec, seed))

print(f"🔢 Pendentes: {len(combinacoes)} (técnica×seed)")
print(f"⏱️  Tempo estimado: ~{len(combinacoes) * 0.5:.0f} min")

resultados_novos = []
pbar = tqdm(combinacoes, desc="Ataques")

for tec, seed in pbar:
    pbar.set_postfix({'tec': tec[:18], 'seed': seed})
    try:
        df_orig, df_anon = carregar_par_orig_anon(tec, seed)
        linhas = executar_todos_ataques(df_orig, df_anon, tec, seed, NIVEIS_AUX)
        resultados_novos.extend(linhas)
    except Exception as e:
        print(f"\n⚠️  Erro em {tec}/seed={seed}: {e}")
        continue

    # Checkpoint
    if resultados_novos:
        df_novos = pd.DataFrame(resultados_novos)
        df_full = pd.concat([df_prev, df_novos], ignore_index=True)
        df_full.to_parquet(CAMINHO_ATAQUES, index=False)

pbar.close()

df_ataques = pd.read_parquet(CAMINHO_ATAQUES)

print(f"\n✅ Loop de ataques concluído!")
print(f"   Total de linhas: {len(df_ataques):,}")
print(f"   Combinações únicas (técnica × seed): {df_ataques[['tecnica','seed']].drop_duplicates().shape[0]}")
print(f"\n📊 Linhas por ataque:")
print(df_ataques.groupby('ataque').size().to_string())

🆕 Sem checkpoint, começando do zero
🔢 Pendentes: 91 (técnica×seed)
⏱️  Tempo estimado: ~46 min


Ataques:   0%|          | 0/91 [00:00<?, ?it/s]


✅ Loop de ataques concluído!
   Total de linhas: 2,184
   Combinações únicas (técnica × seed): 91

📊 Linhas por ataque:
ataque
A1    819
A2    819
A3    364
A4    182


In [2]:
# ============================================================
# Análise estatística + tabelas formatadas para o paper
# (versão auto-suficiente: contém todos os imports necessários)
# ============================================================

# Imports
import os
import numpy as np
import pandas as pd

# Paths (caso o kernel tenha sido resetado)
BASE_OUT_FASE2 = "/content/drive/MyDrive/Mestrado/Resultados_Fase2"
BASE_OUT_FASE3 = "/content/drive/MyDrive/Mestrado/Resultados_Fase3"
CAMINHO_ATAQUES = f"{BASE_OUT_FASE3}/resultados_ataques.parquet"

# Verificar que o drive está montado
if not os.path.exists(CAMINHO_ATAQUES):
    from google.colab import drive
    drive.mount('/content/drive')

df_a = pd.read_parquet(CAMINHO_ATAQUES)
print(f"📊 Análise sobre {len(df_a):,} linhas de ataques\n")
print(f"   Combinações: {df_a[['tecnica','seed']].drop_duplicates().shape[0]}")
print(f"   Técnicas: {sorted(df_a['tecnica'].unique())}\n")

# ────────────────────────────────────────────────────────────────
# TABELA 5 — A1 (Reidentificação por Raio), nível aux=10%
# ────────────────────────────────────────────────────────────────
print("="*80)
print("TABELA 5 — A1 (Reidentificação por Raio), nível aux=10%")
print("Valores: taxa de sucesso (%) mediana sobre 10 seeds")
print("="*80)

a1 = df_a[(df_a['ataque']=='A1') & (df_a['nivel_aux']==0.10)]
piv_a1 = a1.groupby(['tecnica','valor_param'])['valor'].median().unstack()
piv_a1.columns = [f'{int(c)}m' for c in piv_a1.columns]
piv_a1 = piv_a1 * 100  # converter para %
print(piv_a1.round(2).to_string())

# Também imprime para aux=1% e 5% (para tabela completa do paper)
print("\n--- Mesma tabela, nível aux=1% ---")
a1_1 = df_a[(df_a['ataque']=='A1') & (df_a['nivel_aux']==0.01)]
piv_a1_1 = a1_1.groupby(['tecnica','valor_param'])['valor'].median().unstack() * 100
piv_a1_1.columns = [f'{int(c)}m' for c in piv_a1_1.columns]
print(piv_a1_1.round(2).to_string())

print("\n--- Mesma tabela, nível aux=5% ---")
a1_5 = df_a[(df_a['ataque']=='A1') & (df_a['nivel_aux']==0.05)]
piv_a1_5 = a1_5.groupby(['tecnica','valor_param'])['valor'].median().unstack() * 100
piv_a1_5.columns = [f'{int(c)}m' for c in piv_a1_5.columns]
print(piv_a1_5.round(2).to_string())

# ────────────────────────────────────────────────────────────────
# TABELA 6 — A2 (Top-k), nível aux=10%
# ────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TABELA 6 — A2 (Top-k), nível aux=10%")
print("Valores: taxa de sucesso (%) mediana sobre 10 seeds")
print("="*80)

a2 = df_a[(df_a['ataque']=='A2') & (df_a['nivel_aux']==0.10)]
piv_a2 = a2.groupby(['tecnica','valor_param'])['valor'].median().unstack()
piv_a2.columns = [f'k={int(c)}' for c in piv_a2.columns]
piv_a2 = piv_a2 * 100
print(piv_a2.round(2).to_string())

# ────────────────────────────────────────────────────────────────
# TABELA 7 — A3 (Ambiguidade espacial)
# ────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TABELA 7 — A3 (Ambiguidade espacial)")
print("Valores: número médio de candidatos plausíveis (maior = mais privacidade)")
print("="*80)

a3 = df_a[df_a['ataque']=='A3']
piv_a3 = a3.groupby(['tecnica','valor_param'])['valor'].median().unstack()
piv_a3.columns = [f'{int(c)}m' for c in piv_a3.columns]
print(piv_a3.round(1).to_string())

# ────────────────────────────────────────────────────────────────
# TABELA 8 — A4 (Reconstrução temporal)
# ────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("TABELA 8 — A4 (Reconstrução temporal, janela 7 dias)")
print("Valores: erro em metros (maior = mais privacidade)")
print("="*80)

a4_med = df_a[(df_a['ataque']=='A4') & (df_a['metrica']=='erro_mediano_m')]
a4_p95 = df_a[(df_a['ataque']=='A4') & (df_a['metrica']=='erro_p95_m')]
df_a4 = pd.DataFrame({
    'erro_mediano_m': a4_med.groupby('tecnica')['valor'].median(),
    'erro_p95_m':     a4_p95.groupby('tecnica')['valor'].median(),
}).round(0).astype(int)
print(df_a4.to_string())

# ────────────────────────────────────────────────────────────────
# SÍNTESE — Trade-off
# ────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("SÍNTESE — Trade-off Privacidade × Utilidade")
print("="*80)

# Privacidade: 1 - taxa_A1(200m) com aux=10%
priv = 1 - piv_a1['200m'] / 100

# Utilidade: AUC da Fase 2
df_util = pd.read_parquet(f"{BASE_OUT_FASE2}/ic_por_tecnica.parquet")
util = df_util[df_util['metrica']=='auc'].set_index('tecnica')['mediana']

# Junta
df_tradeoff = pd.DataFrame({
    'privacidade (1 - A1@200m)': priv.round(3),
    'utilidade (AUC mediano)':    util.round(3),
}).dropna()

# Adiciona rankings
df_tradeoff['rank_priv'] = df_tradeoff['privacidade (1 - A1@200m)'].rank(ascending=False).astype(int)
df_tradeoff['rank_util'] = df_tradeoff['utilidade (AUC mediano)'].rank(ascending=False).astype(int)

# Score combinado (média dos rankings)
df_tradeoff['rank_combinado'] = (df_tradeoff['rank_priv'] + df_tradeoff['rank_util']) / 2

df_tradeoff = df_tradeoff.sort_values('rank_combinado')
print(df_tradeoff.to_string())

print(f"\n✅ Análise concluída.")
print(f"   Arquivo salvo em: {BASE_OUT_FASE3}/tradeoff_final.parquet")
df_tradeoff.to_parquet(f"{BASE_OUT_FASE3}/tradeoff_final.parquet")

Mounted at /content/drive
📊 Análise sobre 2,184 linhas de ataques

   Combinações: 91
   Técnicas: ['dp_eps_0.1', 'dp_eps_0.5', 'dp_eps_1.0', 'dp_eps_2.0', 'dp_eps_5.0', 'generalizacao_dec2', 'microagregacao_k10', 'microagregacao_k2', 'microagregacao_k5', 'permutacao']

TABELA 5 — A1 (Reidentificação por Raio), nível aux=10%
Valores: taxa de sucesso (%) mediana sobre 10 seeds
                     50m  100m   200m
tecnica                              
dp_eps_0.1          0.00  0.01   0.02
dp_eps_0.5          0.03  0.12   0.46
dp_eps_1.0          0.12  0.46   1.78
dp_eps_2.0          0.46  1.78   6.13
dp_eps_5.0          2.66  8.97  26.39
generalizacao_dec2  1.09  3.05  11.07
microagregacao_k10  0.00  0.03   0.13
microagregacao_k2   0.01  0.03   0.11
microagregacao_k5   0.01  0.04   0.13
permutacao          0.00  0.01   0.05

--- Mesma tabela, nível aux=1% ---
                     50m  100m   200m
tecnica                              
dp_eps_0.1          0.00  0.00   0.04
dp_eps_0.5     